# 9. 프로젝트와 스크립트

1~8편은 Python 스크립트 하나에서 워크파일을 열고 식을 추정해 모델을 풀었습니다. 실제 모형 작업은 오래 가고 여러 사람(과 Claude)이 함께 합니다 —
식은 블록 파일로 나뉘고, 자료는 달마다 새로 오며, 무엇을 언제 어떤 자료로 계산했는지가 남아야 합니다. **프로젝트**는 그것을 폴더 하나로 다룹니다.

- 블록(방정식 파일)과 모형(블록을 끼운 조립) — `equion.toml`
- 원자료 → 정의 → **자료 회차**(만든 자료는 고치지 않는다)
- 계산할 때마다 **작업 기록**(`work/`) — 결론은 기록 이름으로 말한다
- 원자료에서 보고서까지의 순서를 적은 **스크립트**(`.eqs`) — 새 자료가 오면 한 번 돌리고, 그때 그대로 다시 할 수 있다

이 편은 모두 터미널 명령으로 합니다. 사람이 터미널에서 치는 것, Claude가 치는 것, 노트·스크립트에 적는 것이 같은 줄입니다. 출력을 함께 보이려고
8편처럼 Python에서 명령을 부르는 작은 도우미를 씁니다.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
import equion as eqn


def equion(*args, cwd="prj"):
    """터미널의 `equion …`과 같다 — 프로젝트 폴더(cwd)에서, 화면 출력과 종료 코드를 보인다."""
    env = dict(os.environ, PYTHONIOENCODING="utf-8")
    r = subprocess.run([sys.executable, "-m", "equion", *args], cwd=cwd, capture_output=True, text=True,
                       encoding="utf-8", env=env, stdin=subprocess.DEVNULL)
    print((r.stdout + r.stderr).rstrip())
    if r.returncode:
        print(f"(종료 코드 {r.returncode})")
    return r.returncode


equion("new", "prj", "--template", "desk", "--force", cwd=".")

## 9.1 블록과 모형 — 조립

블록은 방정식 파일 하나(`EQUATIONS/demand.txt` …), 식 한 줄은 **부품** `블록:목표`(`demand:CN`)입니다. 모형은 블록과 다른 모형을 이름으로 끼워
만듭니다(`equion.toml`의 `[model.main]` — `include = ["core"]`, `blocks = ["price", "finance"]`). 조립 검사는 계산 없이 내생·외생·겹친 목표·
모르는 변수·외생 자료의 끝을 봅니다.

In [ ]:
print(Path("prj/EQUATIONS/demand.txt").read_text(encoding="utf-8"))
equion("assemble", "main")

## 9.2 자료 회차

원자료(`DATA/raw/` — 받은 그대로)를 정의(`DATA/series.toml`·`derived.txt`)대로 읽은 것이 자료 회차입니다(`work/data/<회차>/`). 만든 회차는 고치지
않고, 새 자료가 오면 새 회차를 만들어 바꿉니다(`data build --use`). 예제 프로젝트는 만들 때 첫 회차를 만들어 두었습니다.

In [ ]:
equion("data", "list")

## 9.3 식 고르기 — 추정·후보·비교·채택

추정은 할 때마다 기록(`est:…`)을 남깁니다. 블록 파일의 식은 그대로 두고 **후보 식**을 추정해(`--for 부품`) 견주고, 바꿔 끼워 과거 재현(HS)으로
확인한 뒤 채택합니다. 채택하면 블록 파일의 그 줄과 채택표(`EQUATIONS/adopted.toml`)가 바뀌고, 그 뒤 모형은 **채택 계수**로 풉니다(다시 추정하지 않는다).

In [ ]:
equion("est", "demand:CN", "--as", "cn1")
equion("est", "CN c Y CN(-1) CN(-2)", "--for", "demand:CN", "--as", "cn2")
equion("compare", "--part", "demand:CN")

판단 기준(`equion.toml`의 `[criteria]` — 기대 부호·진단 p값)은 후보마다 '기준' 줄에 통과·걸러짐과 까닭을 적습니다. 바꿔 끼우기는 파일을 바꾸지 않습니다.

In [ ]:
equion("hs", "main", "--as", "h_now")
equion("hs", "main", "--swap", "demand:CN=est:cn2", "--vs", "hs:h_now", "--as", "h_cn2")

채택과 바뀜 기록 — 정의 파일이 바뀔 때마다 누가·어떤 명령으로·무엇이 바뀌었는지 남고(`history show`), 되돌릴 수 있습니다(`history revert`).
되돌리기는 **그 바뀜 하나만** 물립니다 — 뒤의 바뀜은 그대로 두고, 뒤의 바뀜이 같은 줄을 고쳤으면 아무것도 쓰지 않고 멈춥니다(종료 코드 3).
그래도 그 파일을 그 바뀜 앞 내용으로 통째로 되돌리려면 `--force`(뒤의 바뀜도 그 파일에서 함께 물린다).

In [ ]:
equion("adopt", "est:cn2")
equion("history", "list")

## 9.4 전망과 보고서

전망은 가정 세트(`ASSUMPTIONS/`)·보정항(`ADDFACTORS/`)·시나리오(`SCENARIOS/`)로 풉니다. 계산한 값은 실측 자리에 쓰지 않고 기록에만 남습니다.
보고서에는 명령 블록 없이 결과만 보냅니다 — 표의 모양(행·전년동기비·연간)은 구역 표시에 남아, 새 기록으로 바꿀 때 같은 모양으로 다시 만듭니다.

In [ ]:
equion("sim", "base", "--as", "fc_base")
equion("sim", "oil20", "--as", "fc_oil20")
equion("report", "new", "memo", "--title", "이번 전망")
equion("report", "add", "memo", "sim:fc_base", "--rows", "Y,CN,I", "--show", "yoy", "--annual", "--title", "성장률")
equion("report", "add", "memo", "sim:fc_oil20", "--rows", "Y,P", "--show", "diff", "--title", "유가 20% 상승의 영향")

분석도 같습니다 — 분석 하나 = 명령 하나 = 기록 하나(`an:`). 노트(`notes/*.md`)의 ```` ```equion ```` 블록에 적으면 `note run`이 결과를 그 아래에 채웁니다.

In [ ]:
equion("an", "unitroot", "core", "--trend", "c")

## 9.5 스크립트 한 번 — 새 자료가 오면

지금까지의 순서를 스크립트에 적어 두면(`FLOWS/monthly.eqs` — 한 줄이 명령 하나) 새 자료가 올 때 한 번 돌리면 됩니다. 새 원자료는 받은 상자
`DATA/raw/inbox/`에 놓습니다 — 파일 이름으로 출처를 짝짓고 불러오기 틀로 읽어 봅니다. 여기서는 예제 자료에 한 분기를 더한 합성 표를 씁니다.

In [ ]:
df = pd.read_csv("prj/DATA/raw/toy_q.csv", index_col=0)
df.loc["2020Q1"] = df.iloc[-1] * 1.01
df.to_csv("prj/DATA/raw/inbox/toy_q_2020Q1.csv")
equion("data", "inbox")
equion("check", "FLOWS/monthly.eqs")

실행합니다. 단계마다 한 줄 — `✓` 정상, `!` 사람이 볼 것(판단 기준을 통과하지 못한 식은 옛 계수를 쓰고 계속), `×`이면 멈춥니다. 미리보기는
`--dry-run`, 매개변수는 `--set scenarios=base`, 중간부터는 `--from 단계`.

In [ ]:
equion("run", "FLOWS/monthly.eqs")

실행 기록(`run:monthly_0001`)에는 값을 바꿔 넣은 뒤의 스크립트·정의 사본·받은 원자료·자료 회차·판이 들어 있어, 그때 그대로 다시 돌려 단계마다
숫자를 견줄 수 있습니다. 프로젝트는 바뀌지 않습니다.

In [ ]:
equion("run", "--replay", "run:monthly_0001", "-q")

## 9.6 Claude와 함께

같은 폴더에서 Claude Code를 띄우면 Claude도 같은 명령을 씁니다. 예제 프로젝트의 `CLAUDE.md`가 먼저 `equion guide`(모형 작업 안내 — 규칙·프로젝트
지도·명령·분석·판단 기준·보고 꼴)를 읽게 합니다. 규칙의 핵심은 사람과 같습니다 — 계산은 명령으로, 결론에는 근거 기록의 이름, 결과는 고치지 않고,
정의 파일은 명령(채택·`block add`·`scenario define`·제안 …)으로 먼저, "제안만"이면 파일 대신 `equion propose`.

In [ ]:
print("\n".join(eqn.help.guide_text().splitlines()[4:19]))

Claude Code 스킬 초안도 들어 있습니다(`equion guide skill`). 쓰려면 사람이 시킬 때 `equion guide skill --install user`(모든 프로젝트) 또는
`--install project`(이 프로젝트의 `.claude/skills/`)로 넣습니다.

## 정리

| 할 일 | 명령 |
|---|---|
| 조립 검사 | `equion assemble main` |
| 자료 회차 | `equion data list` · `data build --use` · `data inbox` · `data diff` |
| 식 고르기 | `equion est 부품` · `est "식" --for 부품` · `compare --part 부품` · `hs --swap 부품=est:… --vs hs:…` · `adopt est:…` |
| 전망·보고서 | `equion sim base oil20` · `report new 이름` · `report add 이름 sim:… --rows … --show yoy` |
| 분석·노트 | `equion an <분석> <대상>` · `an keep an:…` · `note run notes/x.md` |
| 기록·바뀜 | `equion work list` · `work show 기록` · `history list` · `history revert h…`(그 바뀜 하나만 — 뒤의 바뀜과 겹치면 멈춘다) |
| 시나리오·이정표 | `equion scenario define 이름 --shock POIL=+20%` · `an split "sim:… sim:…"` · `milestone make 이름` · `bundle make` |
| 스크립트 | `equion check x.eqs` · `run x.eqs [--dry-run · --set · --from]` · `run --replay run:…` · `script new·export --py` |
| Claude | `equion guide` · `guide skill --install user` |

파일 형식은 [파일 형식](../../docs/reference/formats.md), 명령은 [명령줄](../../docs/reference/cli.md), 할 일별 처방은 [레시피](../../docs/howto/README.md)
(R48~R66 — 프로젝트·회차·후보·전망 기록·제안·분석·노트·보고서·스크립트·자료 정의·시나리오 쓰기·요인 나누기·이정표와 묶어 보내기·비선형 시스템)에 있습니다.